# Clustering with KMeans

imports

In [64]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import pandas as pd
import glob
import os


getting the data

In [65]:

# Set the path to your folder
folder_path = '../training_data/'

# Get a list of all CSV files in the folder
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

# Read and concatenate all CSV files
combined_df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

# Display the combined DataFrame
print(combined_df)

                Timestamp  Steps  PAM Score  tabel
0     2025-06-04 14:00:00     15     1.4375    NaN
1     2025-06-04 14:01:00     35     5.5000    NaN
2     2025-06-04 14:02:00      1     0.4375    NaN
3     2025-06-04 14:05:00     35     2.7500    NaN
4     2025-06-04 14:06:00     56     3.9375    NaN
...                   ...    ...        ...    ...
7129  1970-02-01 13:38:00      4     0.7500    NaN
7130  1970-02-01 13:39:00      6     1.0625    NaN
7131  1970-02-01 13:40:00      2     0.3125    NaN
7132  1970-02-01 13:41:00      0     0.0000    NaN
7133  1970-02-01 13:42:00      6     0.5000    NaN

[7134 rows x 4 columns]


In [66]:
len(combined_df)

7134

In [67]:
# combined_df = combined_df.drop_duplicates()
len(combined_df)

7134

cleaning and scaling the data

In [68]:
combined_df = combined_df.drop(columns=['Timestamp'])

# --- (2) Select numeric columns and standardize ---
numeric_df = combined_df.select_dtypes(include='number')
scaler = StandardScaler()
scaled_data = scaler.fit_transform(numeric_df)

In [69]:

# --- (3) (Optional) Plot the Elbow curve to double-check your choice ---
#     If you already know “n_clusters” from a prior run, you can skip this block.
wcss = []
K_range = range(1, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(scaled_data)
    wcss.append(km.inertia_)

plt.figure(figsize=(6, 4))
plt.plot(K_range, wcss, 'o-', linewidth=2)
plt.title('Elbow Method (1 ≤ k ≤ 10)')
plt.xlabel('Number of clusters k')
plt.ylabel('WCSS (inertia)')
plt.xticks(K_range)
plt.grid(True)
plt.show()


ValueError: Input X contains NaN.
KMeans does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

# Inertia
from the inertia we can see that it slows down at 3,
which indicates that we have 3 clusters

In [ ]:
def KMeans_clusters_with_N_clusters(n_clusters):
    # --- Fit KMeans using the user‐specified n_clusters ---
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(scaled_data)
    
    # --- Attach “cluster” labels back into the original DataFrame ---
    clustered_df = combined_df.copy()
    clustered_df['cluster'] = cluster_labels
    
    # If you want a DataFrame of standardized features + cluster ID:
    scaled_df = pd.DataFrame(scaled_data, columns=numeric_df.columns)
    scaled_df['cluster'] = cluster_labels
    
    # --- Compute centroids in original feature units ---
    centroids_std = kmeans.cluster_centers_                      # centroids in standardized space
    centroids_orig = scaler.inverse_transform(centroids_std)     # back to original units
    centroids_df = pd.DataFrame(centroids_orig, columns=numeric_df.columns)
    centroids_df['cluster'] = range(n_clusters)
    
    feat_x = numeric_df.columns[0]   # e.g. first numeric column
    feat_y = numeric_df.columns[1]   # e.g. second numeric column
    
    plt.figure(figsize=(7, 5))
    for cid in range(n_clusters):
        mask = (cluster_labels == cid)
        plt.scatter(
            scaled_data[mask, numeric_df.columns.get_loc(feat_x)],
            scaled_data[mask, numeric_df.columns.get_loc(feat_y)],
            s=30,
            alpha=0.6,
            label=f'Cluster {cid}'
        )
    
    # Plot centroids (in standardized space)
    plt.scatter(
        centroids_std[:, numeric_df.columns.get_loc(feat_x)],
        centroids_std[:, numeric_df.columns.get_loc(feat_y)],
        s=100,
        c='red',
        marker='X',
        edgecolor='k',
        label='Centroids'
    )
    
    plt.title(f'KMeans Clustering (k={n_clusters}) on "{feat_x}" vs "{feat_y}"')
    plt.xlabel(f'{feat_x} (standardized)')
    plt.ylabel(f'{feat_y} (standardized)')
    plt.legend()
    plt.grid(True)
    return plt

In [ ]:

# --- (A) USER SETS THIS: number of clusters to fit ---
n_clusters = 3   # ← change this to 2, 3, 5, etc.
KMeans_clusters_with_N_clusters(n_clusters=n_clusters).show()

# you can see that the data can be optimally split up into 3 clusters

### In the code below you can see more clusters, but these will most likely cause overfitting in the data

In [ ]:
# try out 8 different clusters
for i in range(2,10):
    KMeans_clusters_with_N_clusters(n_clusters=i).show()

In [ ]:
def KMeans_clusters_with_N_clusters(n_clusters):
    # --- Fit KMeans ---
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(scaled_data)

    # --- Attach labels to full original data (including Timestamp) ---
    labeled_df = original_df.copy()
    labeled_df['tabel'] = cluster_labels  # Add the cluster label column

    # --- Export to CSV ---
    output_path = f'clustered_output_k{n_clusters}.csv'
    labeled_df.to_csv(output_path, index=False)
    print(f"Clustered data with timestamps saved to '{output_path}'")

    # --- Optional plot ---
    feat_x = numeric_df.columns[0]
    feat_y = numeric_df.columns[1]
    plt.figure(figsize=(7, 5))
    for cid in range(n_clusters):
        mask = (cluster_labels == cid)
        plt.scatter(
            scaled_data[mask, numeric_df.columns.get_loc(feat_x)],
            scaled_data[mask, numeric_df.columns.get_loc(feat_y)],
            s=30,
            alpha=0.6,
            label=f'Cluster {cid}'
        )

    centroids_std = kmeans.cluster_centers_
    plt.scatter(
        centroids_std[:, numeric_df.columns.get_loc(feat_x)],
        centroids_std[:, numeric_df.columns.get_loc(feat_y)],
        s=100,
        c='red',
        marker='X',
        edgecolor='k',
        label='Centroids'
    )

    plt.title(f'KMeans Clustering (k={n_clusters}) on "{feat_x}" vs "{feat_y}"')
    plt.xlabel(f'{feat_x} (standardized)')
    plt.ylabel(f'{feat_y} (standardized)')
    plt.legend()
    plt.grid(True)
    return plt


In [ ]:
n_clusters = 3
KMeans_clusters_with_N_clusters(n_clusters=n_clusters).show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --- Load clustered data ---
file_path = 'clustered_output_k3.csv'  # change if you used a different filename
df = pd.read_csv(file_path)

# --- Basic check ---
print(df.head())

# --- Check column names (optional) ---
print("Available columns:", df.columns)

# --- Plot by 'Steps' and 'Pam Score' colored by 'tabel' cluster ---
plt.figure(figsize=(8, 6))
for label in sorted(df['tabel'].unique()):
    cluster_data = df[df['tabel'] == label]
    plt.scatter(cluster_data['Steps'], cluster_data['PAM Score'], label=f'Cluster {label}', alpha=0.6)

plt.xlabel('Steps')
plt.ylabel('PAM Score')
plt.title('Clusters on Steps vs Pam Score')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
